# SGLang 多轮对话状态管理 — 实战教程

**定位：** 演示如何使用 SGLang 管理多轮对话状态，包括消息历史维护、上下文窗口控制与对话管理器封装。适合希望构建连续对话应用的开发者。

> **一条命令启动服务：**

```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-0.5B-Instruct --host 127.0.0.1 --port 30000
```

*本 Notebook 为原创示例代码，可自由用于学习、教学及发布到 Gitee 等平台*

---

## 硬件与环境要求

| 项目 | 最低要求 | 推荐配置 |
|------|---------|----------|
| GPU | NVIDIA（支持 CUDA） | RTX 3060 及以上 |
| 显存 (VRAM) | ≥ 6 GB | ≥ 8 GB |
| 驱动版本 | ≥ 525.0 | 最新稳定版 |
| CUDA 版本 | ≥ 12.1 | 12.4+ |
| Python | 3.9 – 3.12 | 3.10 |
| 操作系统 | Linux / WSL2 | Ubuntu 22.04 |

---

## 依赖安装

推荐使用虚拟环境隔离依赖：

```bash
# 第一步：创建并激活虚拟环境
python -m venv sglang-env
source sglang-env/bin/activate

# 第二步：安装 SGLang 及所需依赖
pip install "sglang[all]" "openai>=1.0.0" "requests>=2.28.0"
```

---

## 使用指南

1. 运行「环境检查」单元格，确认 GPU 和 CUDA 可用
2. 如需在 Notebook 中安装依赖，运行「可选安装」代码单元格
3. 运行「启动 SGLang 服务」单元格，等待服务就绪
4. 按顺序运行多轮对话相关单元格，体验对话状态管理
5. 完成后运行「停止服务释放显存」单元格清理资源

---

In [ ]:
import sys  # 导入系统模块
print(f"Python 版本: {sys.version}")  # 打印 Python 版本信息

import torch  # 导入 PyTorch 深度学习框架
print(f"PyTorch 版本: {torch.__version__}")  # 打印 PyTorch 版本
print(f"CUDA 是否可用: {torch.cuda.is_available()}")  # 检查 CUDA 是否可用

if torch.cuda.is_available():  # 如果 CUDA 可用
    print(f"CUDA 版本: {torch.version.cuda}")  # 打印 CUDA 版本号
    print(f"GPU 名称: {torch.cuda.get_device_name(0)}")  # 打印第一张 GPU 名称
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3  # 计算总显存（GB）
    print(f"GPU 显存: {vram_gb:.1f} GB")  # 打印显存大小
else:  # CUDA 不可用
    print("⚠️ 未检测到 GPU，请检查驱动和 CUDA 安装")  # 提示用户检查环境

In [ ]:
import subprocess  # 导入子进程模块
import sys  # 导入系统模块

subprocess.check_call([  # 执行 pip 安装命令
    sys.executable, "-m", "pip", "install", "-q",  # 使用当前 Python 解释器静默安装
    "sglang[all]", "openai>=1.0.0", "requests>=2.28.0"  # 安装 SGLang 及所需依赖
])  # 安装命令执行完成
print("✅ 依赖安装完成")  # 打印安装成功提示

## 多轮对话的核心：维护消息历史

多轮对话的关键在于**维护完整的消息历史列表**。每次调用 API 时，需要将之前所有的对话消息一起发送给模型，这样模型才能理解上下文并给出连贯的回复。

消息历史的基本结构：
```python
messages = [
    {"role": "system", "content": "系统提示词"},
    {"role": "user", "content": "用户第一轮问题"},
    {"role": "assistant", "content": "模型第一轮回答"},
    {"role": "user", "content": "用户第二轮问题"},
    {"role": "assistant", "content": "模型第二轮回答"},
    ...
]
```

**注意**：随着对话轮数增加，消息列表会越来越长，可能超出模型的上下文窗口。后面我们将演示如何处理这个问题。

## 启动 SGLang 服务

在后台启动 SGLang 推理服务，并等待服务就绪。首次运行需下载模型，可能需要数分钟。

In [ ]:
import subprocess  # 导入子进程模块
import time  # 导入时间模块
import requests  # 导入 HTTP 请求库
import sys  # 导入系统模块

server_process = subprocess.Popen(  # 以子进程方式启动 SGLang 推理服务
    [  # 命令参数列表
        sys.executable, "-m", "sglang.launch_server",  # 调用 SGLang 服务启动模块
        "--model-path", "Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型路径（首次自动下载）
        "--host", "127.0.0.1",  # 监听本机回环地址
        "--port", "30000",  # 指定服务端口号
    ],  # 命令参数结束
    stdout=subprocess.PIPE,  # 捕获标准输出
    stderr=subprocess.PIPE,  # 捕获标准错误输出
)  # 子进程创建完成
print(f"SGLang 服务进程已启动，PID = {server_process.pid}")  # 打印进程 ID

max_wait = 600  # 最大等待时间 600 秒（含模型下载时间）
start_time = time.time()  # 记录开始时间
ready = False  # 服务就绪标志初始化为 False

while time.time() - start_time < max_wait:  # 在超时范围内持续轮询
    try:  # 尝试连接服务
        r = requests.get("http://127.0.0.1:30000/v1/models", timeout=5)  # 向模型列表接口发请求
        if r.status_code == 200:  # 返回 200 表示服务已就绪
            print(f"✅ SGLang 服务就绪！耗时 {time.time() - start_time:.1f} 秒")  # 打印就绪信息
            ready = True  # 更新就绪标志
            break  # 退出轮询循环
    except requests.ConnectionError:  # 连接失败说明服务尚未启动完成
        pass  # 忽略异常，继续等待
    time.sleep(2)  # 每隔 2 秒重试一次

if not ready:  # 如果超时仍未就绪
    print("❌ 服务启动超时，请检查 GPU 显存或查看进程日志")  # 打印超时错误信息

## 基础多轮对话

手动维护消息列表，逐轮发送请求并记录回复，演示最基本的多轮对话实现方式。

In [ ]:
from openai import OpenAI  # 导入 OpenAI 客户端库

client = OpenAI(  # 创建 OpenAI 兼容客户端实例
    base_url="http://127.0.0.1:30000/v1",  # 指向本地 SGLang 服务地址
    api_key="EMPTY",  # 本地服务无需真实 API Key
)  # 客户端创建完成

messages = [  # 初始化消息历史列表
    {"role": "system", "content": "你是一个知识渊博的AI助手，擅长用简洁的语言解释技术概念。"}  # 系统角色提示
]  # 消息列表初始化完成

# ---- 第一轮对话 ----
user_msg_1 = "什么是机器学习？"  # 第一轮用户提问
messages.append({"role": "user", "content": user_msg_1})  # 将用户消息追加到历史
print(f"👤 用户: {user_msg_1}")  # 打印用户问题

resp_1 = client.chat.completions.create(  # 发送聊天补全请求
    model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称
    messages=messages,  # 传入完整消息历史
    max_tokens=256,  # 限制最大生成 token 数
)  # 请求完成
assistant_msg_1 = resp_1.choices[0].message.content  # 提取模型回复内容
messages.append({"role": "assistant", "content": assistant_msg_1})  # 将模型回复追加到历史
print(f"🤖 助手: {assistant_msg_1}\n")  # 打印模型回复

# ---- 第二轮对话 ----
user_msg_2 = "它和深度学习有什么区别？"  # 第二轮用户追问（依赖上下文）
messages.append({"role": "user", "content": user_msg_2})  # 将用户消息追加到历史
print(f"👤 用户: {user_msg_2}")  # 打印用户问题

resp_2 = client.chat.completions.create(  # 发送第二轮请求
    model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称
    messages=messages,  # 传入包含前两轮的完整历史
    max_tokens=256,  # 限制最大生成 token 数
)  # 请求完成
assistant_msg_2 = resp_2.choices[0].message.content  # 提取模型回复内容
messages.append({"role": "assistant", "content": assistant_msg_2})  # 将模型回复追加到历史
print(f"🤖 助手: {assistant_msg_2}\n")  # 打印模型回复

# ---- 第三轮对话 ----
user_msg_3 = "给我举个实际例子"  # 第三轮追问（继续依赖上下文）
messages.append({"role": "user", "content": user_msg_3})  # 将用户消息追加到历史
print(f"👤 用户: {user_msg_3}")  # 打印用户问题

resp_3 = client.chat.completions.create(  # 发送第三轮请求
    model="Qwen/Qwen2.5-0.5B-Instruct",  # 指定模型名称
    messages=messages,  # 传入包含前三轮的完整历史
    max_tokens=256,  # 限制最大生成 token 数
)  # 请求完成
assistant_msg_3 = resp_3.choices[0].message.content  # 提取模型回复内容
messages.append({"role": "assistant", "content": assistant_msg_3})  # 将模型回复追加到历史
print(f"🤖 助手: {assistant_msg_3}\n")  # 打印模型回复

# ---- 打印完整对话历史 ----
print("=" * 50)  # 打印分隔线
print("完整对话历史：")  # 打印标题
for i, msg in enumerate(messages):  # 遍历所有消息
    role_label = {"system": "📋 系统", "user": "👤 用户", "assistant": "🤖 助手"}.get(msg["role"], msg["role"])  # 为每个角色映射可读标签
    print(f"  [{i}] {role_label}: {msg['content'][:80]}...")  # 打印消息序号、角色和前80字符

## 封装多轮对话管理器

为了方便复用，我们将多轮对话逻辑封装为 `ChatSession` 类。该类自动维护消息历史，并支持可配置的最大历史轮数，防止上下文窗口溢出。

In [ ]:
from openai import OpenAI  # 导入 OpenAI 客户端库


class ChatSession:  # 定义多轮对话管理器类
    """多轮对话管理器：自动维护消息历史并支持上下文截断"""

    def __init__(  # 初始化方法
        self,  # 实例自身引用
        base_url="http://127.0.0.1:30000/v1",  # SGLang 服务地址，默认本地
        api_key="EMPTY",  # API 密钥，本地服务使用占位符
        model="Qwen/Qwen2.5-0.5B-Instruct",  # 默认使用的模型名称
        system_prompt="你是一个有帮助的AI助手。",  # 默认系统提示词
        max_history=20,  # 最大保留的消息条数（不含 system）
        max_tokens=512,  # 每次回复的最大 token 数
    ):  # 参数定义结束
        self.client = OpenAI(base_url=base_url, api_key=api_key)  # 创建 OpenAI 兼容客户端
        self.model = model  # 保存模型名称
        self.system_prompt = system_prompt  # 保存系统提示词
        self.max_history = max_history  # 保存最大历史条数
        self.max_tokens = max_tokens  # 保存最大 token 数
        self.messages = [  # 初始化消息列表
            {"role": "system", "content": self.system_prompt}  # 首条消息为系统提示
        ]  # 消息列表初始化完成

    def send(self, user_msg):  # 发送用户消息并获取回复
        """发送用户消息，返回助手回复文本"""
        self.messages.append({"role": "user", "content": user_msg})  # 将用户消息追加到历史
        self._truncate_history()  # 如有需要，截断过长的历史

        response = self.client.chat.completions.create(  # 调用聊天补全 API
            model=self.model,  # 使用配置的模型
            messages=self.messages,  # 传入当前消息历史
            max_tokens=self.max_tokens,  # 设置最大生成长度
        )  # API 调用完成
        assistant_msg = response.choices[0].message.content  # 提取回复文本
        self.messages.append({"role": "assistant", "content": assistant_msg})  # 将回复追加到历史
        self._truncate_history()  # 再次检查并截断历史
        return assistant_msg  # 返回助手回复文本

    def _truncate_history(self):  # 私有方法：截断历史消息
        """保留 system 消息 + 最近 max_history 条对话消息"""
        non_system = self.messages[1:]  # 取出除 system 之外的所有消息
        if len(non_system) > self.max_history:  # 如果超出最大历史限制
            trimmed = non_system[-self.max_history:]  # 只保留最近的 max_history 条
            self.messages = [self.messages[0]] + trimmed  # 重组：system + 截断后的消息

    def reset(self):  # 重置对话历史
        """清空对话历史，只保留 system 消息"""
        self.messages = [  # 重新初始化消息列表
            {"role": "system", "content": self.system_prompt}  # 仅保留系统提示
        ]  # 重置完成
        print("🔄 对话已重置")  # 打印重置确认信息

    def get_history(self):  # 获取当前对话历史
        """返回当前消息历史列表的副本"""
        return list(self.messages)  # 返回消息列表的浅拷贝

    def get_turn_count(self):  # 获取当前对话轮数
        """返回已完成的对话轮数"""
        user_count = sum(1 for m in self.messages if m["role"] == "user")  # 统计用户消息数量
        return user_count  # 返回轮数（等于用户消息数）


print("✅ ChatSession 类定义完成")  # 打印类定义成功提示

## 使用对话管理器

实例化 `ChatSession` 并进行多轮对话，演示自动消息管理功能。

In [ ]:
session = ChatSession(  # 创建对话会话实例
    system_prompt="你是一位Python编程专家，擅长用简洁的语言讲解编程概念。",  # 设定系统角色为编程专家
    max_history=10,  # 最多保留最近 10 条消息
    max_tokens=256,  # 每次回复最多 256 个 token
)  # 会话实例创建完成

questions = [  # 定义多轮对话的问题列表
    "Python 中的列表和元组有什么区别？",  # 第一轮问题
    "那什么时候应该用元组而不是列表？",  # 第二轮追问
    "能举一个用元组做字典键的例子吗？",  # 第三轮追问
    "如果我要存储坐标点，用哪种数据结构好？",  # 第四轮追问
]  # 问题列表定义完成

for i, question in enumerate(questions, 1):  # 遍历所有问题，序号从 1 开始
    print(f"--- 第 {i} 轮 ---")  # 打印当前轮次标题
    print(f"👤 用户: {question}")  # 打印用户问题
    reply = session.send(question)  # 发送问题并获取回复
    print(f"🤖 助手: {reply}\n")  # 打印助手回复

print(f"总对话轮数: {session.get_turn_count()}")  # 打印总对话轮数
print(f"当前历史消息条数: {len(session.get_history())}")  # 打印历史消息总数（含 system）

## 上下文窗口管理：自动截断

当对话轮数过多时，消息历史会超出模型的上下文窗口。`ChatSession` 的 `max_history` 参数可以自动截断旧消息，只保留最近的对话。

下面演示设置一个很小的 `max_history`，观察截断行为。

In [ ]:
short_session = ChatSession(  # 创建一个历史限制很短的会话
    system_prompt="你是一个友好的AI助手。",  # 设定简单的系统提示
    max_history=4,  # 只保留最近 4 条消息（2轮对话）
    max_tokens=128,  # 每次回复最多 128 个 token
)  # 短历史会话创建完成

test_questions = [  # 准备超过限制的多轮问题
    "我叫小明，请记住我的名字",  # 第一轮：告知名字
    "今天天气怎么样？",  # 第二轮：无关话题
    "你觉得学编程难吗？",  # 第三轮：又一个无关话题
    "我之前告诉你我叫什么名字？",  # 第四轮：测试是否记得第一轮内容
]  # 测试问题定义完成

for i, question in enumerate(test_questions, 1):  # 遍历所有测试问题
    print(f"--- 第 {i} 轮 ---")  # 打印当前轮次
    print(f"👤 用户: {question}")  # 打印用户问题
    reply = short_session.send(question)  # 发送问题并获取回复
    print(f"🤖 助手: {reply}")  # 打印助手回复
    history = short_session.get_history()  # 获取当前消息历史
    print(f"   📊 当前历史条数: {len(history)}（含 system）")  # 打印历史条数
    print(f"   📝 历史中的角色: {[m['role'] for m in history]}\n")  # 打印历史中各消息的角色

print("=" * 50)  # 打印分隔线
print("⚠️ 注意：由于 max_history=4，第一轮的消息已被截断。")  # 提示截断行为
print("模型可能无法回忆起早期对话内容，这是上下文管理的预期行为。")  # 解释截断的影响
print("\n最终历史内容：")  # 打印最终历史标题
for msg in short_session.get_history():  # 遍历最终的消息历史
    role_label = {"system": "📋 系统", "user": "👤 用户", "assistant": "🤖 助手"}.get(msg["role"], msg["role"])  # 映射角色标签
    content_preview = msg["content"][:60] + ("..." if len(msg["content"]) > 60 else "")  # 截取内容预览
    print(f"  {role_label}: {content_preview}")  # 打印角色和内容预览

## 停止服务释放显存

In [ ]:
import os  # 导入操作系统模块
import signal  # 导入信号处理模块

if 'server_process' in dir() and server_process.poll() is None:  # 检查服务进程是否仍在运行
    os.kill(server_process.pid, signal.SIGTERM)  # 向服务进程发送终止信号
    server_process.wait(timeout=10)  # 等待进程退出，最多等 10 秒
    print(f"✅ SGLang 服务已停止 (PID={server_process.pid})")  # 打印停止成功信息
else:  # 服务进程不存在或已结束
    print("ℹ️ 服务进程未运行或已结束")  # 打印提示信息

if 'torch' in dir() and torch.cuda.is_available():  # 检查 PyTorch 和 CUDA 是否可用
    torch.cuda.empty_cache()  # 清空 GPU 显存缓存
    print("✅ GPU 显存缓存已清理")  # 打印显存清理完成信息

## 常见问题排查

### 1. 上下文过长错误（Context Too Long）

**现象**：调用 API 时报错 `This model's maximum context length is ...`

**原因**：多轮对话中消息历史累积过长，超出模型上下文窗口。

**解决**：
- 减小 `ChatSession` 的 `max_history` 参数，只保留最近的几轮对话
- 减小 `max_tokens` 以留出更多空间给历史消息
- 启动服务时增大 `--max-model-len` 参数（需要更多显存）

### 2. 模型遗忘早期对话内容

**现象**：模型无法回忆起早期对话中提到的内容。

**原因**：上下文已被自动截断，早期消息已不在发送给模型的消息列表中。

**解决**：
- 增大 `max_history` 参数以保留更多历史
- 启动服务时增大 `--max-model-len`（如 `--max-model-len 4096`）以支持更长上下文
- 在 system prompt 中总结关键信息，使模型始终可见重要上下文

---

**结语**：本教程演示了 SGLang 多轮对话的核心机制——消息历史管理。通过 `ChatSession` 类可以方便地构建具有记忆能力的对话应用，同时通过 `max_history` 参数避免上下文溢出。建议根据实际模型的上下文长度和业务需求调整参数。